# SleepStageNet CNN+BiLSTM — Kaggle P100 Training

Run 20-fold LOSO-CV on Kaggle's **P100 GPU** (free tier: 30 hrs/week, 12hr sessions).

**Runtime:** Select **GPU P100** accelerator: Settings → Accelerator → GPU P100

**Persistence:** Output is saved to `/kaggle/working/` which persists after the session ends (up to 20GB).
Combined with `--skip_existing`, you can start a new session and resume from where you left off.

In [ ]:
# 1. Configuration — edit these paths/settings as needed
import os

REPO_DIR = "/kaggle/working/deepsleepnet-lite"

# Output persists in /kaggle/working/ after session ends (up to 20GB).
# Combined with --skip_existing, you can restart a session and resume from where it left off.
OUTPUT_DIR = "/kaggle/working/temporal_output"

# Training hyperparameters
SEQ_LEN = 5         # Sequence length (should be odd: 3, 5, 11, 21)
BATCH_SIZE = 64      # Batch size
FOLDS = list(range(20))  # Which folds to train (range(20) for full LOSO-CV)

# Data: upload the preprocessed Sleep-EDF zip as a Kaggle dataset,
# or use gdown to fetch from Google Drive.
DATA_FILE_ID = "1wDu9tl6_P250k522tQC9LUUVh7ocG1_x"  # Google Drive file ID
DATA_DIR = f"{REPO_DIR}/data/SleepEDF/processed/eeg_FpzCz_PzOz_v1"

REPO_URL = "https://github.com/manishdas/deepsleepnet-lite.git"

In [ ]:
# 2. Install deps, clone repo
!pip install -q scikit-learn gdown

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")
    !cd {REPO_DIR} && git pull

In [ ]:
# 3. Download data (skip if already exists)
import glob

eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))

if len(eeg_files) >= 39:
    print(f"Data already exists: {len(eeg_files)} files")
else:
    !gdown {DATA_FILE_ID} -O /tmp/data.zip
    !cd {REPO_DIR} && unzip -q -o /tmp/data.zip
    eeg_files = glob.glob(os.path.join(DATA_DIR, 'SC4*E0.npz'))
    print(f"Downloaded: {len(eeg_files)} files")

In [ ]:
# 4. Verify GPU
import torch, multiprocessing

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'CPU cores: {multiprocessing.cpu_count()}')

In [ ]:
# 5. Sanity check — 2 epochs per stage (quick validation before full run)
%cd {REPO_DIR}/temporal
!python train_sequence.py --fold 0 --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
    --cnn_epochs 2 --lstm_epochs 2 --finetune_epochs 2 \
    --output_dir {OUTPUT_DIR}/sanity_check

In [ ]:
# 6. Full training (skip_existing for crash recovery across sessions)
for fold in FOLDS:
    !python train_sequence.py --fold {fold} --seq_len {SEQ_LEN} --batch_size {BATCH_SIZE} \
        --output_dir {OUTPUT_DIR} --skip_existing

In [ ]:
# 7. Generate plots (saved to OUTPUT_DIR/figures/ for download)
!pip install -q matplotlib

FIGURES_DIR = f"{OUTPUT_DIR}/figures"

!python plot_results.py --mode all \
    --results_dir {OUTPUT_DIR} \
    --output_dir {FIGURES_DIR} \
    --data_dir {DATA_DIR}

# Show the generated figures inline
from IPython.display import Image, display
import glob
for fig in sorted(glob.glob(f'{FIGURES_DIR}/*.png')):
    print(f'\n{os.path.basename(fig)}')
    display(Image(filename=fig))

In [ ]:
# 8. Verify results and list all output files for download
import json, glob, numpy as np

results_files = sorted(glob.glob(f'{OUTPUT_DIR}/fold*_results.json'))
print(f'Completed folds: {len(results_files)} / {len(FOLDS)}')
print(f'Model checkpoints: {len(glob.glob(f"{OUTPUT_DIR}/*.pt"))}')

if results_files:
    accs, f1s, kappas = [], [], []
    for fp in results_files:
        with open(fp) as f:
            r = json.load(f)
        accs.append(r['test_metrics']['accuracy'])
        f1s.append(r['test_metrics']['f1_macro'])
        kappas.append(r['test_metrics']['kappa'])
    print(f'\nTest accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}')
    print(f'Test F1 macro:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
    print(f'Test kappa:     {np.mean(kappas):.4f} ± {np.std(kappas):.4f}')

# List everything in output dir for download
print(f'\n=== Files in {OUTPUT_DIR} ===')
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'  {os.path.relpath(path, OUTPUT_DIR):50s} {size_mb:.1f} MB')